In [24]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

import numpy as np
import os
import plotly.io as pio
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy.sparse as sp
from anndata import AnnData

In [25]:
class PolygonAnnotator:
    def __init__(self, image, cmap='hot'):
        self.image = image
        self.coords = None
        self.done = False

        self.fig, self.ax = plt.subplots()
        self.ax.imshow(image, cmap=cmap)
        self.ax.set_title("Click to draw polygon. Double-click to finish.")

        self.selector = PolygonSelector(
            self.ax,
            self.onselect,
            useblit=True,
            props=dict(color='cyan', linewidth=2),
            handle_props=dict(marker='o', markersize=5, mec='cyan', mfc='cyan')
        )

        self.fig.canvas.mpl_connect('close_event', self._on_close)
        plt.show(block=True)  # Blocks until the window is closed

    def _on_close(self, event):
        self.done = True

    def onselect(self, verts):
        self.coords = verts
        print(f"Polygon drawn with {len(verts)} points.")
        plt.close(self.fig)  # Automatically close the window

    def get_mask(self):
        if not self.coords:
            print("No polygon was drawn.")
            return None
        ny, nx = self.image.shape
        y, x = np.mgrid[:ny, :nx]
        points = np.vstack((x.flatten(), y.flatten())).T
        path = Path(self.coords)
        return path.contains_points(points).reshape((ny, nx))

In [26]:
def get_mz_heatmap_image(adata, target_mz, tol=0.1):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values.astype(int)
    y = adata.obs["y"].values.astype(int)

    width = x.max() + 1
    height = y.max() + 1
    image = np.zeros((height, width))
    image[y, x] = intensities

    return image, matched_mz

In [27]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad


In [28]:
def sort_adata_by_mz(adata, mz_column="mzs"):
    def to_float_with_warning(index, label="var_names"):
        mzs = pd.to_numeric(index, errors="coerce")
        n_nans = mzs.isna().sum()
        if n_nans > 0:
            print(f"⚠️ Warning: {n_nans} entries in {label} could not be converted to float and were set to NaN.")
        return mzs
    adata.var[mz_column] = to_float_with_warning(adata.var_names)
    adata = adata[:, adata.var[mz_column].sort_values().index]
    return adata


In [29]:
adata = sort_adata_by_mz(adata, mz_column="mzs")
adata.var

,mzs
136.0150,136.0150
136.0608,136.0608
137.0251,137.0251
138.0287,138.0287
139.0359,139.0359
...,...
1520.1588,1520.1588
1521.1633,1521.1633
1522.1682,1522.1682
1542.1475,1542.1475


In [30]:
# ---- Run ----

target_mz = 788.5
image, matched_mz = get_mz_heatmap_image(adata, target_mz)
print(f"Target m/z: {target_mz} → Matched m/z: {matched_mz}")

custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

#custom_cmap = "binary"
# Use the annotator with your image and colormap
annotator = PolygonAnnotator(image, cmap=custom_cmap)
mask = annotator.get_mask()

if mask is not None:
    masked_image = np.where(mask, image, 0)
    plt.imshow(masked_image, cmap=custom_cmap)
    plt.title(f"Masked m/z = {matched_mz:.4f}")
    plt.show()
else:
    print("No mask created.")

Target m/z: 788.5 → Matched m/z: 788.5027
Polygon drawn with 28 points.


In [31]:
# Get spatial coordinates
x = adata.obs["x"].astype(int).values
y = adata.obs["y"].astype(int).values

# Mask value at each spot
adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)
adata.obs["tissue"] = adata.obs["tissue"].astype(int)  # 1 = inside, 0 = outside
print(adata.obs["tissue"].value_counts())

tissue
0    9409
1    7196
Name: count, dtype: int64


/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_32490/1596079847.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)


In [32]:
adata1 = adata
adata.obs["tissue"]

0        0
1        0
2        0
3        0
4        0
        ..
16600    0
16601    0
16602    0
16603    0
16604    0
Name: tissue, Length: 16605, dtype: int64

In [33]:
adata = adata1
adata

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [34]:
def filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=2.0, layer=None, normalize_tic=True):
    # 1. Extract raw data
    data = adata.layers[layer] if layer is not None else adata.X

    # 2. TIC normalization per pixel
    if normalize_tic:
        # Sum across features for each observation (axis=1)
        tic = np.asarray(data.sum(axis=1)).ravel()
        # Avoid division by zero
        tic[tic == 0] = 1.0
        # Normalize in place (if sparse, convert to array)
        data = data / tic[:, None]

    # 3. Define tissue masks
    mask_tissue = adata.obs[tissue_key] == tissue_val
    mask_non    = adata.obs[tissue_key] == non_tissue_val

    # 4. Compute mean normalized intensity per feature
    mean_tissue    = np.asarray(data[mask_tissue].mean(axis=0)).ravel()
    mean_nontissue = np.asarray(data[mask_non].mean(axis=0)).ravel()

    # 5. Identify features to remove
    to_remove = mean_tissue < foldchange * mean_nontissue

    # 6. Print removed m/z names
    mz_to_remove = adata.var_names[to_remove]
    print(f"Dropping {len(mz_to_remove)} m/z features:", mz_to_remove.values)

    # 7. Subset and return
    return adata[:, ~to_remove].copy()

In [35]:
foldchange = 3

# Filter m/z features based on tissue mask
adata = filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=foldchange, layer=None, normalize_tic=True)

Dropping 659 m/z features: ['136.0150' '137.0251' '138.0287' '139.0359' '154.0255' '155.0350'
 '156.0424' '159.0070' '160.0107' '163.0405' '165.0200' '172.0540'
 '173.0578' '176.0111' '177.0146' '178.0269' '179.0362' '181.0163'
 '187.0399' '189.0757' '190.0676' '191.0379' '192.9801' '195.0905'
 '196.0981' '196.9856' '197.1082' '197.9919' '199.0008' '199.0439'
 '199.9998' '200.0492' '201.0571' '202.0613' '202.1793' '203.0369'
 '203.0680' '203.1800' '204.0398' '210.9897' '211.0595' '213.0560'
 '213.9619' '214.0577' '214.9722' '215.0298' '216.0426' '217.0514'
 '218.0560' '219.0630' '219.9751' '220.0723' '220.9800' '226.1111'
 '227.0378' '228.0455' '229.0025' '229.0488' '230.0543' '231.0686'
 '232.0653' '233.0507' '233.0774' '235.9458' '239.0343' '240.0412'
 '241.0502' '242.0613' '243.0269' '243.0677' '244.0353' '245.0458'
 '246.0516' '247.0594' '248.0624' '249.0743' '251.0421' '254.0257'
 '255.0287' '256.0338' '257.0478' '258.0498' '259.0608' '259.2436'
 '260.0386' '261.0393' '262.0491' '

In [36]:
adata

AnnData object with n_obs × n_vars = 16605 × 341
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import entropy, variation, skew, kurtosis
from libpysal.weights import KNN
from esda.moran import Moran
from esda.geary import Geary
import matplotlib.pyplot as plt
from tqdm import tqdm


def compute_spatial_entropy(intensities):
    intensities = np.array(intensities, dtype=float)
    intensities = intensities[intensities > 0]
    if len(intensities) == 0:
        return np.nan
    p = intensities / intensities.sum()
    return entropy(p)


def compute_morans_I(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Moran(intensities, w).I
    except Exception:
        return np.nan


def compute_gearys_C(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Geary(intensities, w).C
    except Exception:
        return np.nan


def compute_center_of_mass(x_coords, y_coords, intensities):
    total_intensity = np.sum(intensities)
    if total_intensity == 0:
        return np.nan, np.nan
    x_center = np.sum(x_coords * intensities) / total_intensity
    y_center = np.sum(y_coords * intensities) / total_intensity
    return x_center, y_center


def analyze_metrics(adata, k_neighbors=8):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    rows = []
    tissue_centroid_x = adata.obs.loc[adata.obs["tissue"] == True, "x"].mean()
    tissue_centroid_y = adata.obs.loc[adata.obs["tissue"] == True, "y"].mean()

    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        for flag, region in zip([True, False], ["tissue", "background"]):
            subset = adata[adata.obs["tissue"] == flag]
            if subset.shape[0] == 0:
                row[f"moran_I_{region}"] = np.nan
                row[f"geary_C_{region}"] = np.nan
                row[f"entropy_{region}"] = np.nan
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan
                row[f"kurtosis_{region}"] = np.nan
                row[f"region_size"] = 0 if region == "tissue" else np.nan
                continue

            intensities = subset.X[:, mz_index].toarray().flatten() if hasattr(subset.X, "toarray") else subset.X[:, mz_index].flatten()
            x = subset.obs["x"].values
            y = subset.obs["y"].values

            nonzero_intensities = intensities[intensities > 0]

            row[f"moran_I_{region}"] = compute_morans_I(x, y, intensities, k=k_neighbors)
            row[f"geary_C_{region}"] = compute_gearys_C(x, y, intensities, k=k_neighbors)
            row[f"entropy_{region}"] = compute_spatial_entropy(intensities)

            if len(nonzero_intensities) > 0:
                row[f"cv_{region}"] = variation(nonzero_intensities)
                row[f"skew_{region}"] = skew(nonzero_intensities)
                row[f"kurtosis_{region}"] = kurtosis(nonzero_intensities)
            else:
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan
                row[f"kurtosis_{region}"] = np.nan

            if region == "tissue":
                row[f"region_size"] = np.sum(intensities > 0)

        # Center of mass and its distance from tissue centroid
        all_x = adata.obs["x"].values
        all_y = adata.obs["y"].values
        all_intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()
        x_com, y_com = compute_center_of_mass(all_x, all_y, all_intensities)
        if not np.isnan(x_com) and not np.isnan(y_com):
            dist_to_tissue_centroid = np.sqrt((x_com - tissue_centroid_x)**2 + (y_com - tissue_centroid_y)**2)
        else:
            dist_to_tissue_centroid = np.nan

        row["com_distance_to_tissue"] = dist_to_tissue_centroid

        rows.append(row)

    df_metrics = pd.DataFrame(rows)
    return df_metrics


In [38]:
# ---------- Analysis ----------
df_spatial = analyze_metrics(adata, k_neighbors=8)


Processing m/z features: 100%|██████████| 341/341 [06:35<00:00,  1.16s/it]


In [39]:
df_spatial

,mz,moran_I_tissue,geary_C_tissue,entropy_tissue,cv_tissue,skew_tissue,kurtosis_tissue,region_size,moran_I_background,geary_C_background,entropy_background,cv_background,skew_background,kurtosis_background,com_distance_to_tissue
0,136.0608,0.251014,0.742324,8.711782,0.623616,2.005792,8.077647,7189,0.496775,0.502123,8.672406,1.117923,5.471074,49.287430,4.341580
1,161.1338,0.212052,0.787520,8.685532,0.602369,0.990047,1.310533,7060,0.710451,0.290221,8.793441,0.764328,0.875136,0.239841,25.476267
2,180.0717,0.223795,0.771375,8.783014,0.449051,0.864520,1.313566,7196,0.745643,0.256671,8.979751,0.577271,0.587494,-0.203503,20.202910
3,184.0720,0.168107,0.829506,8.720886,0.635747,2.695173,13.902505,7196,0.529450,0.459114,8.744621,1.109302,3.894505,24.070905,5.047282
4,185.0686,0.148600,0.847663,8.692978,0.649311,1.799155,5.888100,7178,0.342391,0.656844,9.015736,0.520752,0.952728,2.067640,14.209809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,1520.1588,0.292777,0.701401,8.678668,0.664777,1.839228,8.658307,7161,0.593025,0.379231,7.948772,1.487311,4.150705,26.207740,1.689382
337,1521.1633,0.294535,0.702695,8.666640,0.673561,1.501778,3.743212,7142,0.595946,0.378053,7.956549,1.441267,3.967170,22.662962,2.036688
338,1522.1682,0.300892,0.694668,8.669024,0.680791,1.947594,10.734394,7160,0.585907,0.385600,7.959302,1.438577,3.872188,24.523910,1.854764
339,1542.1475,0.241904,0.759584,8.684505,0.643374,1.618243,6.319945,7149,0.513589,0.455544,8.093106,1.161464,3.006125,13.140249,0.733016


In [49]:
def compute_dispersion_metrics(adata, threshold=0.01):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    tissue_centroid_x = adata.obs.loc[adata.obs["tissue"] == True, "x"].mean()
    tissue_centroid_y = adata.obs.loc[adata.obs["tissue"] == True, "y"].mean()

    results = []

    for mz_index in tqdm(range(num_mz), desc="Computing dispersion metrics"):
        mz = mz_values[mz_index]
        row = {"mz": mz}

        intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()
        x = adata.obs["x"].values
        y = adata.obs["y"].values
        tissue_mask = adata.obs["tissue"].values

        # Global stats
        row["mean_intensity"] = np.mean(intensities)
        row["sum_intensity"] = np.sum(intensities)

        # Background vs. tissue
        tissue_intensity = intensities[tissue_mask]
        background_intensity = intensities[~tissue_mask]
        row["mean_tissue"] = np.mean(tissue_intensity)
        row["mean_background"] = np.mean(background_intensity)
        row["sum_tissue"] = np.sum(tissue_intensity)
        row["sum_background"] = np.sum(background_intensity)
        row["background_fraction"] = row["sum_background"] / (row["sum_tissue"] + row["sum_background"] + 1e-10)
        row["SBR"] = row["mean_tissue"] / (row["mean_background"] + 1e-10)

        # Center of mass
        x_com, y_com = compute_center_of_mass(x, y, intensities)
        if not np.isnan(x_com) and not np.isnan(y_com):
            row["com_distance"] = np.sqrt((x_com - tissue_centroid_x)**2 + (y_com - tissue_centroid_y)**2)
        else:
            row["com_distance"] = np.nan

        # Max distance of signal above threshold
        above_threshold_mask = intensities > threshold
        if np.any(above_threshold_mask):
            distances = np.sqrt((x[above_threshold_mask] - tissue_centroid_x)**2 + (y[above_threshold_mask] - tissue_centroid_y)**2)
            row["max_dispersion_distance"] = np.max(distances)
            row["mean_dispersion_distance"] = np.mean(distances)
        else:
            row["max_dispersion_distance"] = np.nan
            row["mean_dispersion_distance"] = np.nan

        results.append(row)

    return pd.DataFrame(results)


In [55]:
# Merge spatial_stats and dispersion metrics tables on 'mz'
df_dispersion = compute_dispersion_metrics(adata, threshold=0.00001)


Computing dispersion metrics: 100%|██████████| 341/341 [00:00<00:00, 4116.47it/s]


In [56]:
df_dispersion

,mz,mean_intensity,sum_intensity,mean_tissue,mean_background,sum_tissue,sum_background,background_fraction,SBR,com_distance,max_dispersion_distance,mean_dispersion_distance
0,136.0608,224.911472,3734655.0,35.131466,25.766998,583358.0,427861.0,0.423114,1.363429e+00,4.341580,97.946329,48.824581
1,161.1338,170.795303,2836056.0,44.764288,112.268353,743311.0,1864216.0,0.714936,3.987258e-01,25.476267,97.946329,49.438436
2,180.0717,642.279374,10665049.0,108.631015,655.097922,1803818.0,10877901.0,0.857762,1.658241e-01,20.202910,97.946329,49.802818
3,184.0720,2438.292803,40487852.0,90.572960,186.135983,1503964.0,3090788.0,0.672678,4.865957e-01,5.047282,97.946329,49.795819
4,185.0686,249.710991,4146451.0,37.601084,356.496838,624366.0,5919630.0,0.904589,1.054738e-01,14.209809,97.946329,49.812769
...,...,...,...,...,...,...,...,...,...,...,...,...
336,1520.1588,53.478290,888007.0,11.332731,0.433363,188180.0,7196.0,0.036832,2.615064e+01,1.689382,97.227408,44.391612
337,1521.1633,47.718880,792372.0,11.899368,3.033544,197589.0,50372.0,0.203145,3.922596e+00,2.036688,97.256474,44.319184
338,1522.1682,51.754592,859385.0,4.533092,0.000000,75272.0,0.0,0.000000,4.533092e+10,1.854764,97.256474,44.293340
339,1542.1475,41.800662,694100.0,6.067088,0.000000,100744.0,0.0,0.000000,6.067088e+10,0.733016,97.256474,44.302854


In [57]:
df_combined

,mz,moran_I_tissue,geary_C_tissue,entropy_tissue,cv_tissue,skew_tissue,kurtosis_tissue,region_size,moran_I_background,geary_C_background,...,sum_intensity,mean_tissue,mean_background,sum_tissue,sum_background,background_fraction,SBR,com_distance,max_dispersion_distance,mean_dispersion_distance
0,136.0150,0.402909,0.563836,8.423925,0.937218,3.612106,29.216555,6263,0.472251,0.521698,...,7371791.0,325.086119,821.651912,5398055.0,1.364353e+07,0.716512,3.956494e-01,21.081225,97.987830,50.864515
1,136.0608,0.274174,0.717313,8.711960,0.630430,1.996617,7.665990,7213,0.484973,0.504957,...,3734655.0,35.041855,25.782897,581870.0,4.281250e+05,0.423888,1.359112e+00,4.776256,97.987830,48.875635
2,137.0251,0.504461,0.437524,8.638821,0.913475,7.718039,126.556753,7220,0.715651,0.281514,...,428154643.0,13216.062632,69263.522132,219452720.0,1.150121e+09,0.839766,1.908084e-01,30.599717,97.987830,49.831380
3,138.0287,0.479846,0.470469,8.615226,0.888083,5.549950,71.305765,7191,0.693563,0.302819,...,33900021.0,1393.288768,5353.160494,23135560.0,8.888923e+07,0.793478,2.602741e-01,28.045596,97.987830,49.870530
4,139.0359,0.348665,0.629341,8.552695,0.804669,3.156242,24.766977,6702,0.641089,0.354863,...,7500120.0,243.696778,1122.429389,4046585.0,1.863794e+07,0.821615,2.171155e-01,25.421182,97.987830,50.443243
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1520.1588,0.282129,0.710055,8.682300,0.665134,1.907717,9.838545,7185,0.602611,0.372853,...,888007.0,11.303824,0.434809,187700.0,7.220000e+03,0.037041,2.599723e+01,1.766917,97.273310,44.398883
996,1521.1633,0.289467,0.705970,8.668574,0.677301,1.573787,4.472077,7164,0.603301,0.375785,...,792372.0,11.869015,3.043662,197085.0,5.054000e+04,0.204099,3.899584e+00,2.187706,97.293417,44.325654
997,1522.1682,0.288431,0.706904,8.673340,0.680326,2.043889,12.512857,7185,0.592800,0.384028,...,859385.0,4.521530,0.000000,75080.0,0.000000e+00,0.000000,4.521530e+10,2.046541,97.293417,44.303681
998,1542.1475,0.236942,0.764670,8.688594,0.641846,1.616562,6.364922,7173,0.520480,0.450817,...,694100.0,6.087323,0.000000,101080.0,0.000000e+00,0.000000,6.087323e+10,0.360461,97.293417,44.305560


In [58]:
# Use the correct column names from the provided DataFrame sample
corrected_features = [
    "entropy_background", "moran_I_background", "geary_C_background",
    "background_fraction", "SBR", "com_distance", 
    "max_dispersion_distance", "mean_dispersion_distance",
    "mean_intensity", "sum_intensity"
]

# Normalize features
scaler = StandardScaler()
normalized_values = scaler.fit_transform(df_combined_clean[corrected_features])
df_normalized = pd.DataFrame(normalized_values, columns=[f"z_{col}" for col in corrected_features])

# Combine normalized values with original DataFrame
df_combined_normalized = pd.concat([df_combined_clean.reset_index(drop=True), df_normalized], axis=1)

# Create composite score for ranking
df_combined_normalized["dispersion_score"] = (
    df_normalized["z_entropy_background"] +
    df_normalized["z_background_fraction"] +
    df_normalized["z_com_distance"] +
    df_normalized["z_max_dispersion_distance"] +
    df_normalized["z_mean_dispersion_distance"] -
    df_normalized["z_mean_intensity"] * 0.5
)

# Rank by score
df_ranked = df_combined_normalized.sort_values("dispersion_score", ascending=False).reset_index(drop=True)
df_ranked.head(10)


,mz,moran_I_tissue,geary_C_tissue,entropy_tissue,cv_tissue,skew_tissue,kurtosis_tissue,region_size,moran_I_background,geary_C_background,...,z_moran_I_background,z_geary_C_background,z_background_fraction,z_SBR,z_com_distance,z_max_dispersion_distance,z_mean_dispersion_distance,z_mean_intensity,z_sum_intensity,dispersion_score
0,883.0213,0.054355,0.924739,6.943257,0.834527,2.778817,16.522586,1372,0.388366,0.609526,...,-0.846408,0.868745,0.771877,-0.116345,0.829646,0.271319,2.596750,-0.178302,-0.178302,4.693847
1,875.0211,0.257542,0.683799,7.018284,1.248540,11.361487,200.235462,1605,0.463966,0.533451,...,-0.337754,0.358283,0.689328,-0.116345,0.987341,0.271319,2.376430,-0.156172,-0.156172,4.642567
2,876.0159,0.075689,0.908567,7.045875,0.821140,2.594906,11.427591,1503,0.416145,0.581069,...,-0.659509,0.677799,0.662703,-0.116345,0.985065,0.271319,2.448812,-0.174250,-0.174250,4.630636
3,877.0113,0.059936,0.928358,7.231247,0.801190,1.942181,5.724948,1816,0.355091,0.643269,...,-1.070291,1.095165,1.005657,-0.116345,0.788635,0.271319,2.223104,-0.180153,-0.180153,4.592454
4,1065.9956,0.116391,0.861671,7.027066,0.928753,3.598687,21.963658,1537,0.346456,0.651157,...,-1.128392,1.148093,0.836202,-0.116345,0.677973,0.271319,2.417494,-0.179928,-0.179928,4.476600
5,1033.0245,0.109597,0.874409,7.001704,0.888549,4.020563,32.805629,1466,0.398008,0.598946,...,-0.781538,0.797758,0.765804,-0.116345,0.608877,0.271319,2.527224,-0.179318,-0.179318,4.455989
6,1068.0066,0.049180,0.937811,6.930367,0.781205,2.007346,6.803494,1326,0.325102,0.673003,...,-1.272068,1.294675,0.615674,-0.116345,0.700595,0.271319,2.609293,-0.183019,-0.183019,4.418623
7,1052.0283,0.079330,0.905655,6.734458,0.836511,2.537492,12.308705,1115,0.366502,0.631735,...,-0.993520,1.017769,0.102532,-0.116345,0.796584,0.271319,2.820970,-0.181943,-0.181943,4.186684
8,853.0236,0.089532,0.885405,7.067776,0.820957,2.196631,7.848525,1551,0.327402,0.670944,...,-1.256597,1.280861,0.356514,-0.116345,0.850043,0.271319,2.428630,-0.182651,-0.182651,4.166548
9,874.0140,0.173719,0.802870,7.224829,0.953806,4.653403,44.011161,1887,0.431240,0.566532,...,-0.557943,0.580258,0.480033,-0.116345,0.899711,0.271319,2.169270,-0.169437,-0.169437,4.157374


In [59]:
df_ranked.head(350)


,mz,moran_I_tissue,geary_C_tissue,entropy_tissue,cv_tissue,skew_tissue,kurtosis_tissue,region_size,moran_I_background,geary_C_background,...,z_moran_I_background,z_geary_C_background,z_background_fraction,z_SBR,z_com_distance,z_max_dispersion_distance,z_mean_dispersion_distance,z_mean_intensity,z_sum_intensity,dispersion_score
0,883.0213,0.054355,0.924739,6.943257,0.834527,2.778817,16.522586,1372,0.388366,0.609526,...,-0.846408,0.868745,0.771877,-0.116345,0.829646,0.271319,2.596750,-0.178302,-0.178302,4.693847
1,875.0211,0.257542,0.683799,7.018284,1.248540,11.361487,200.235462,1605,0.463966,0.533451,...,-0.337754,0.358283,0.689328,-0.116345,0.987341,0.271319,2.376430,-0.156172,-0.156172,4.642567
2,876.0159,0.075689,0.908567,7.045875,0.821140,2.594906,11.427591,1503,0.416145,0.581069,...,-0.659509,0.677799,0.662703,-0.116345,0.985065,0.271319,2.448812,-0.174250,-0.174250,4.630636
3,877.0113,0.059936,0.928358,7.231247,0.801190,1.942181,5.724948,1816,0.355091,0.643269,...,-1.070291,1.095165,1.005657,-0.116345,0.788635,0.271319,2.223104,-0.180153,-0.180153,4.592454
4,1065.9956,0.116391,0.861671,7.027066,0.928753,3.598687,21.963658,1537,0.346456,0.651157,...,-1.128392,1.148093,0.836202,-0.116345,0.677973,0.271319,2.417494,-0.179928,-0.179928,4.476600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,394.0619,0.357322,0.629507,8.565044,0.770481,1.980506,6.135956,6753,0.656783,0.340046,...,0.959575,-0.939457,0.655338,-0.116345,0.578377,0.271319,-0.232790,-0.105568,-0.105568,1.927933
346,393.0658,0.470661,0.512314,8.618201,0.762999,2.231514,8.103466,7044,0.728607,0.267409,...,1.442830,-1.426853,0.623638,-0.116345,0.720379,0.271319,-0.335565,-0.037160,-0.037160,1.925971
347,269.0502,0.461798,0.480038,8.525104,1.074403,10.515345,214.351344,6930,0.555183,0.443121,...,0.275981,-0.247825,0.556603,-0.116345,1.217078,0.271319,-0.295460,0.356978,0.356978,1.925887
348,163.0405,0.109404,0.881103,8.555031,0.682935,1.442968,3.260314,6422,0.491427,0.506076,...,-0.152988,0.174598,0.576085,-0.116345,0.566592,0.271319,-0.129260,-0.127788,-0.127788,1.920676


In [60]:
df_combined_normalized

,mz,moran_I_tissue,geary_C_tissue,entropy_tissue,cv_tissue,skew_tissue,kurtosis_tissue,region_size,moran_I_background,geary_C_background,...,z_moran_I_background,z_geary_C_background,z_background_fraction,z_SBR,z_com_distance,z_max_dispersion_distance,z_mean_dispersion_distance,z_mean_intensity,z_sum_intensity,dispersion_score
0,136.0150,0.402909,0.563836,8.423925,0.937218,3.612106,29.216555,6263,0.472251,0.521698,...,-0.282011,0.279421,0.331426,-0.116345,0.261225,0.271319,-0.082651,-0.043401,-0.043401,1.547346
1,136.0608,0.274174,0.717313,8.711960,0.630430,1.996617,7.665990,7213,0.484973,0.504957,...,-0.196414,0.167091,-1.008962,-0.116345,-1.715967,0.271319,-0.686551,-0.119389,-0.119389,-3.488700
2,137.0251,0.504461,0.437524,8.638821,0.913475,7.718039,126.556753,7220,0.715651,0.281514,...,1.355652,-1.332208,0.895998,-0.116345,1.415468,0.271319,-0.396351,8.747658,8.747658,-1.767961
3,138.0287,0.479846,0.470469,8.615226,0.888083,5.549950,71.305765,7191,0.693563,0.302819,...,1.207039,-1.189255,0.683975,-0.116345,1.105747,0.271319,-0.384463,0.510831,0.510831,1.961027
4,139.0359,0.348665,0.629341,8.552695,0.804669,3.156242,24.766977,6702,0.641089,0.354863,...,0.853983,-0.840036,0.812856,-0.116345,0.787502,0.271319,-0.210566,-0.040720,-0.040720,2.295452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1520.1588,0.282129,0.710055,8.682300,0.665134,1.907717,9.838545,7185,0.602611,0.372853,...,0.595093,-0.719327,-2.780948,-0.116345,-2.080890,-1.983649,-2.045864,-0.178861,-0.178861,-11.515230
996,1521.1633,0.289467,0.705970,8.668574,0.677301,1.573787,4.472077,7164,0.603301,0.375785,...,0.599731,-0.699655,-2.015725,-0.116345,-2.029863,-1.920192,-2.068099,-0.180859,-0.180859,-10.602772
997,1522.1682,0.288431,0.706904,8.673340,0.680326,2.043889,12.512857,7185,0.592800,0.384028,...,0.529076,-0.644341,-2.950616,1.539531,-2.046981,-1.920192,-2.074771,-0.179459,-0.179459,-11.572306
998,1542.1475,0.236942,0.764670,8.688594,0.641846,1.616562,6.364922,7173,0.520480,0.450817,...,0.042488,-0.196187,-2.950616,2.112956,-2.251441,-1.920192,-2.074200,-0.182912,-0.182912,-11.356690
